## Script for annotating random samples

In [38]:
# imports
import re
import pandas as pd
from IPython.display import display, HTML
from ast import literal_eval

In [39]:
df = pd.read_csv('article_sample.csv')
df

,index,title,body,ai_related,company_hits,matched_keywords_all
0,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,no,[],[]
1,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",no,[],[]
2,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,no,[],['kunstmatige intelligentie']
3,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,no,[],[]
4,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,no,['facebook'],[]
...,...,...,...,...,...,...
115,1886,Digitale agenda verdient meer ambitie Digitale...,Voor het eerst heeft een Nederlands kabinet ee...,yes,"['facebook', 'google']","['ai', 'artificial intelligence', 'kunstmatige..."
116,22315,Voor het Radboudumc is dit de meest bijzondere...,Een bijzondere foto met een bijzonder verhaal....,no,[],['ai']
117,18212,Love & peacemet aandeelhouders in Silicon Vall...,"Een heerlijke nieuwe toekomst, waarin alle ras...",yes,"['facebook', 'google', 'microsoft', 'tesla']",['artificiële intelligentie']
118,13354,"'Het zijn Trump, China en Poetin die Europa wa...",ABSTRACT\nInterviewÉlie Cohen Econoom\nDe were...,yes,"['asml', 'microsoft']","['ai', 'kunstmatige intelligentie']"


In [40]:

# --- helper: highlight text ---
def highlight_keywords(text, keywords):
    """Highlights keywords in text with bold, larger, green font."""
    if pd.isna(text):
        return ""
    if not isinstance(keywords, (set, list)):
        raise ValueError("Keywords must be a set or list")
    if len(keywords) == 0:
        return str(text)

    pattern = r'\b(?:' + '|'.join(re.escape(str(word)) for word in keywords if str(word).strip()) + r')\b'

    def replace_keyword(match):
        kw = match.group(0)
        return f'<span style="font-size:1.3em; font-weight:bold; color:#009900;">{kw}</span>'

    return re.sub(pattern, replace_keyword, str(text), flags=re.IGNORECASE)


# --- helper: safely parse lists ---
def _parse_listish(val):
    """Converts possible formats (list, stringified list, csv string) into a list of strings."""
    if isinstance(val, (list, set, tuple)):
        return [str(x) for x in val]
    if isinstance(val, str) and val.strip():
        try:
            parsed = literal_eval(val)
            if isinstance(parsed, (list, set, tuple)):
                return [str(x) for x in parsed]
        except Exception:
            pass
        return [s.strip() for s in val.split(",") if s.strip()]
    return []


# --- main viewer function ---
def display_article_range(
    csv_path="sample_120_articles.csv",
    start=0,
    end=10,
    title_col="title",
    body_col="body",
    ai_col="ai_related",
    matched_col_candidates=("matched_keywords", "matched_keywords_all"),
    company_col="company_hits"
):
    """
    Display a specific range of articles from the CSV file with highlighted AI keywords and company hits.

    Parameters
    ----------
    csv_path : str
        Path to CSV file.
    start, end : int
        Row positions to display (preserves the CSV’s original order).
        E.g., start=20, end=30 → shows rows 20–29.
    """
    df = pd.read_csv(csv_path)

    # --- sanity checks ---
    required = {"index", title_col, body_col, ai_col}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"CSV missing required column(s): {missing}")

    # detect keyword column
    mk_col = next((c for c in matched_col_candidates if c in df.columns), None)

    # --- ensure valid range and preserve order ---
    start = max(0, start)
    end = min(end, len(df))
    if start >= end:
        print("⚠️ Invalid range or empty selection.")
        return pd.DataFrame()

    # select exact range in original order
    subset = df.iloc[start:end].copy()

    for _, row in subset.iterrows():
        # collect highlight terms: matched keywords + company hits
        row_terms = set()
        if mk_col:
            row_terms.update(x.lower() for x in _parse_listish(row.get(mk_col)))
        row_terms.update(x.lower() for x in _parse_listish(row.get(company_col, [])))

        # highlight
        title_h = highlight_keywords(row[title_col], row_terms)
        body_h  = highlight_keywords(row[body_col],  row_terms)

        # header
        header = (
            f"<div style='margin:0.5em 0;'>"
            f"<strong>Row:</strong> {row.name} &nbsp;|&nbsp; "
            f"<strong>Index:</strong> {row['index']} &nbsp;|&nbsp; "
            f"<strong>ai_related:</strong> {row[ai_col]}"
        )
        if mk_col:
            header += f" &nbsp;|&nbsp; <strong>matched_keywords:</strong> {row.get(mk_col)}"
        if company_col in subset.columns:
            header += f" &nbsp;|&nbsp; <strong>company_hits:</strong> {row.get(company_col)}"
        header += "</div>"

        display(HTML(header))
        display(HTML(f"<h3 style='margin:0.2em 0;'>{title_h}</h3>"))
        display(HTML(f"<div style='line-height:1.5;'>{body_h}</div>"))
        display(HTML("<hr>"))

    print(f"Displayed rows {start}–{end-1} from '{csv_path}' (original order preserved).")
    return subset


In [41]:
# --- encode ai_related: yes→1, no→2 ---
df['ai_related'] = df['ai_related'].map({'yes': 1, 'no': 2})
df

,index,title,body,ai_related,company_hits,matched_keywords_all
0,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,2,[],[]
1,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",2,[],[]
2,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,2,[],['kunstmatige intelligentie']
3,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,2,[],[]
4,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,2,['facebook'],[]
...,...,...,...,...,...,...
115,1886,Digitale agenda verdient meer ambitie Digitale...,Voor het eerst heeft een Nederlands kabinet ee...,1,"['facebook', 'google']","['ai', 'artificial intelligence', 'kunstmatige..."
116,22315,Voor het Radboudumc is dit de meest bijzondere...,Een bijzondere foto met een bijzonder verhaal....,2,[],['ai']
117,18212,Love & peacemet aandeelhouders in Silicon Vall...,"Een heerlijke nieuwe toekomst, waarin alle ras...",1,"['facebook', 'google', 'microsoft', 'tesla']",['artificiële intelligentie']
118,13354,"'Het zijn Trump, China en Poetin die Europa wa...",ABSTRACT\nInterviewÉlie Cohen Econoom\nDe were...,1,"['asml', 'microsoft']","['ai', 'kunstmatige intelligentie']"


In [42]:
df

,index,title,body,ai_related,company_hits,matched_keywords_all
0,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,2,[],[]
1,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",2,[],[]
2,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,2,[],['kunstmatige intelligentie']
3,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,2,[],[]
4,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,2,['facebook'],[]
...,...,...,...,...,...,...
115,1886,Digitale agenda verdient meer ambitie Digitale...,Voor het eerst heeft een Nederlands kabinet ee...,1,"['facebook', 'google']","['ai', 'artificial intelligence', 'kunstmatige..."
116,22315,Voor het Radboudumc is dit de meest bijzondere...,Een bijzondere foto met een bijzonder verhaal....,2,[],['ai']
117,18212,Love & peacemet aandeelhouders in Silicon Vall...,"Een heerlijke nieuwe toekomst, waarin alle ras...",1,"['facebook', 'google', 'microsoft', 'tesla']",['artificiële intelligentie']
118,13354,"'Het zijn Trump, China en Poetin die Europa wa...",ABSTRACT\nInterviewÉlie Cohen Econoom\nDe were...,1,"['asml', 'microsoft']","['ai', 'kunstmatige intelligentie']"


In [ ]:

# --- create XLSX for Joly (includes ai_related) ---
# df[['index', 'ai_related']].to_excel('annotator_joly.xlsx', index=False)
# print("✅ Saved 'annotator_joly.xlsx' with index + ai_related columns.")

# --- create XLSX for Dennis (excludes ai_related) ---
df[['index']].to_excel('annotator_dennis.xlsx', index=False)
print("✅ Saved 'annotator_dennis.xlsx' without ai_related column.")


✅ Saved 'annotator_joly.xlsx' with index + ai_related columns.
✅ Saved 'annotator_dennis.xlsx' without ai_related column.


In [37]:
# Show articles n through k from your sample CSV
display_article_range("article_sample.csv", start=0, end=10)



Displayed rows 0–9 from 'article_sample.csv' (original order preserved).


,index,title,body,ai_related,company_hits,matched_keywords_all
0,17844,Pijnlijke docu stoot Adolescence van de troon ...,Tim Hofman benoemde het al in zijn aflevering ...,no,[],[]
1,3463,Aangepast plaatsingssysteem bevalt leerlingen ...,"Ouders, leerlingen en onderwijsinstellingen zi...",no,[],[]
2,2427,'Academische boycot kan Israël hard in de port...,De Israëlische economie komt in zwaar weer doo...,no,[],['kunstmatige intelligentie']
3,3408,'Tol Duitsland discrimineert buitenlanders' 'T...,Analyse: 'Infrastructuurtoeslag' blijft omstre...,no,[],[]
4,7048,Jos Collignon is anti-Brexit – dus erover teke...,De Brexitsaga is al bijna drie jaar aan de gan...,no,['facebook'],[]
5,13332,Wetenschap Begrip van het bewustzijn begint ni...,In het interview met natuurkundige Cristiane d...,no,[],['neurale netwerken']
6,14857,Sprong vooruit of in de afgrond Sprong vooruit...,ABSTRACT\nFintech\nJonge bedrijven die nieuwe ...,no,"['adyen', 'apple', 'facebook']",[]
7,13227,Hoe Ghibli deel werd van de AI-wapenwedloop Ho...,ABSTRACT\nCLOSE-UP\nVOLLEDIGE TEKST:\nDe gelie...,yes,['google'],"['ai', 'kunstmatige intelligentie', 'openai']"
8,19375,De kansen voor ondernemers op TikTok De kansen...,"Kans 1\nIs jouw doelgroep Generatie Z, geboren...",no,['facebook'],[]
9,27008,‘Met games kan ik mensen in een verhaalwereld ...,"Nu hij zich toelegt op animaties en games, zit...",no,[],[]
